# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step walkthrough for loading, exploring, and processing the FAIR^2 dataset (logistic regression outcomes from Kenyan pastoralist adoption studies) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List available record sets, their fields, and their `@id` values. This is crucial to identify how to programmatically refer to each item in the next steps.

In [ ]:
# Display all record sets and their fields using @id for unambiguous references
print('Available record sets:')
for record_set in dataset.record_sets:
    print(f"- RecordSet name: {getattr(record_set, 'name', 'Unnamed')}   @id: {record_set.id}")
    print('  Fields:')
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field: {getattr(field, 'name', 'Unnamed')}   @id: {field.id}")
    print('')

## 3. Data Extraction

Load data from the available record set(s) into Pandas DataFrames using only their `@id` fields.
* Replace the `record_sets` list as needed depending on which record sets you want to extract (see above output).

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print('No record data found in any record set.')

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps using columns and fields referenced by their `@id`.
* Filter numeric fields, normalize data, and group by categorical fields.
* Replace field `@id`s as appropriate (see list printed above for actual dataset structure).
* **Note:** As a demonstration, the first numeric column found will be used.

In [ ]:
# Try to select a record set and field for EDA
# List available DataFrames (record sets)
if dataframes:
    # Pick the first dataframe with at least one numeric field
    found = False
    for rs_id, df in dataframes.items():
        # Search for numeric columns
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                record_set_id = rs_id
                numeric_field_id = col  # column is the field @id
                found = True
                break
        if found:
            break

    if found:
        print(f"Using record set: {record_set_id}, numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        # Filter
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by another field if available (e.g., the first string/categorical field)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
                group_field = col
                break

        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field detected in any record set for EDA.")
else:
    print('No DataFrames are available for EDA.')

## 5. Visualization

Visualize distributions or relationships. Update field `@id`s below as needed to match those present in your selected DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: If EDA above succeeded, plot the numeric field distribution and group means
if dataframes and 'record_set_id' in locals() and 'numeric_field_id' in locals():
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id} in {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values()
        group_means.plot(kind='bar')
        plt.title(f'Mean of {numeric_field_id} by {group_field}')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print('No valid DataFrame available to plot.')

## 6. Conclusion

This notebook demonstrates loading metadata and records from a FAIR-structured Croissant dataset using `mlcroissant`, referencing all elements by their persistent `@id`. With the steps above, you can:
* Enumerate all available record sets and fields.
* Load tabular record data by `@id` (ensuring reproducibility).
* Filter and transform fields for EDA (e.g., removing outliers, normalization, grouping).
* Create basic visualizations.

This approach fosters robust, reproducible data science workflows for structured datasets in the Croissant format.